In [1]:
# notebooks/06_ltv_optimizer.ipynb — CELL 1

import pandas as pd
import numpy as np

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt

import joblib
import json
import os
import warnings

warnings.filterwarnings('ignore')

# -----------------------------------
# CREATE PLOTS FOLDER
# -----------------------------------
os.makedirs('../plots', exist_ok=True)

# -----------------------------------
# LOAD DATASET
# -----------------------------------
df = pd.read_csv('../data/fd_loans_engineered.csv')

# -----------------------------------
# LOAD XGBOOST MODEL
# -----------------------------------
xgb = joblib.load('../models/xgb_model.pkl')

metrics = json.load(
    open('../models/xgb_metrics.json')
)

FEATURES = metrics['features']

# -----------------------------------
# PREDICT DEFAULT PROBABILITY
# -----------------------------------
df['pred_proba'] = xgb.predict_proba(
    df[FEATURES]
)[:, 1]

# -----------------------------------
# LTV BAND RISK ANALYSIS
# -----------------------------------
ltv_band_risk = df.groupby('ltv_band_num').agg(
    count=('loan_id', 'count'),

    actual_default=('defaulted', 'mean'),

    pred_default=('pred_proba', 'mean'),

    avg_ltv=('ltv', 'mean')
).round(4)

print("\nLTV BAND RISK ANALYSIS:\n")

print(ltv_band_risk)

# -----------------------------------
# RISK THRESHOLD
# -----------------------------------
RISK_THRESHOLD = 0.05

# -----------------------------------
# FUNCTION:
# FIND MAX SAFE LTV
# -----------------------------------
def find_max_safe_ltv(
    customer_row,
    xgb,
    features,
    threshold=0.05
):

    best_ltv = 0.50

    for ltv_test in np.arange(0.50, 0.95, 0.01):

        row = customer_row.copy()

        # Update LTV
        row['ltv'] = ltv_test

        # Recalculate dependent features
        row['loan_amt'] = int(
            row['fd_amount'] * ltv_test
        )

        row['high_ltv'] = int(
            ltv_test > 0.80
        )

        row['ltv_band_num'] = min(
            int((ltv_test - 0.40) / 0.10),
            4
        )

        row['fd_coverage'] = (
            row['fd_amount']
            /
            (row['loan_amt'] + 1)
        )

        # Convert to dataframe
        feat_df = pd.DataFrame([row])[features]

        # Predict risk
        prob = xgb.predict_proba(
            feat_df
        )[0][1]

        # Check threshold
        if prob <= threshold:
            best_ltv = ltv_test
        else:
            break

    return round(best_ltv, 2)

# -----------------------------------
# SAMPLE CUSTOMERS
# -----------------------------------
sample = df.sample(
    1000,
    random_state=42
).copy()

print("\nComputing optimal LTV for 1000 customers...")

sample['max_safe_ltv'] = sample.apply(
    lambda row: find_max_safe_ltv(
        row,
        xgb,
        FEATURES,
        RISK_THRESHOLD
    ),
    axis=1
)

# -----------------------------------
# SUMMARY STATS
# -----------------------------------
print(f"\nAverage Max Safe LTV: {sample['max_safe_ltv'].mean():.3f}")

print("\nDistribution:\n")

print(
    sample['max_safe_ltv']
    .describe()
    .round(3)
)

# -----------------------------------
# PLOT DISTRIBUTION
# -----------------------------------
plt.figure(figsize=(10, 6))

plt.hist(
    sample['max_safe_ltv'],
    bins=20,
    edgecolor='black',
    alpha=0.8
)

plt.title(
    'Distribution of Maximum Safe LTV',
    fontsize=14,
    fontweight='bold'
)

plt.xlabel('Maximum Safe LTV')

plt.ylabel('Customer Count')

plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    '../plots/ltv_optimizer.png',
    dpi=150,
    bbox_inches='tight'
)

plt.close()

print("\nSaved: plots/ltv_optimizer.png")

# -----------------------------------
# SAVE RISK BAND TABLE
# -----------------------------------
ltv_band_risk.to_csv(
    '../data/ltv_risk_bands.csv'
)

print("Saved: data/ltv_risk_bands.csv")


LTV BAND RISK ANALYSIS:

              count  actual_default  pred_default  avg_ltv
ltv_band_num                                              
0             14540          0.0646        0.3093   0.5003
1              7349          0.0680        0.3450   0.6509
2              7236          0.0705        0.3458   0.7508
3              7309          0.1129        0.4699   0.8505
4              3566          0.1565        0.5510   0.9251

Computing optimal LTV for 1000 customers...

Average Max Safe LTV: 0.504

Distribution:

count    1000.000
mean        0.504
std         0.036
min         0.500
25%         0.500
50%         0.500
75%         0.500
max         0.850
Name: max_safe_ltv, dtype: float64

Saved: plots/ltv_optimizer.png
Saved: data/ltv_risk_bands.csv
